In [1]:
import os
import xarray as xr
import xclim
import xsdba
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print(xclim.__version__)
print(xsdba.__version__)

0.58.1
0.5.0


In [2]:
# --- config ---
obs_path = "/mnt/Data/TraCE-Sahul/tasmax/TraCE-Sahul_1500_1990_tasmax.nc"
hist = "/mnt/Data/CMIP6/historical/tasmax/tasmax_ACCESS-ESM1-5_historical_r1i1p1f1_185001-201412.nc"
ssp = "/mnt/Data/CMIP6/ssp370/tasmax/tasmax_ACCESS-ESM1-5_ssp370_r1i1p1f1_201501-210012.nc"
var = "tasmax"
ssp_base = os.path.basename(ssp)
parts = ssp_base.replace(".nc", "").split("_")
var, model, expid, variant, timerange = parts
tstart = "199001"
tend = timerange.split("-")[1]
out_dir = "/mnt/Data/CMIP6/bias_corrected"
out_name = f"{var}_{model}_{expid}_{variant}_{tstart}-{tend}.nc"
out_path = os.path.join(out_dir, out_name)

ds = xr.open_dataset(obs_path, decode_times=False)
print(ds.lat.values[:5])
print(ds.lat.values[-5:])
print(ds.lon.values[:5])
print(ds.lon.values[-5:])

# spatial subset for testing
tas_lon = slice(144, 149)
tas_lat = slice(-44, -40) 

[-44.975 -44.925 -44.875 -44.825 -44.775]
[11.275 11.325 11.375 11.425 11.475]
[105.025 105.075 105.125 105.175 105.225]
[161.275 161.325 161.375 161.425 161.475]


In [3]:
# --- load data ---
## observation data
obs_clim = xr.open_dataset(obs_path, decode_times=False)[var].chunk({'lat':200, "lon":200})
obs_clim = obs_clim.rio.write_crs("EPSG:4326") if hasattr(obs_clim, "rio") else obs_clim
time_index = pd.date_range(start="1500-01-01", periods=obs_clim.sizes["time"], freq="MS", unit="s") + pd.Timedelta(days=15).as_unit("s")
obs_clim = obs_clim.assign_coords(time=time_index)
obs_clim = obs_clim.sel(time=obs_clim.time >= pd.Timestamp("1940-01-01"))

## CMIP6 historical climatology data
mod_clim = xr.open_dataset(hist, decode_times=False)[var].chunk({'lat':200, "lon":200})
time_index = pd.date_range(start="1850-01-01", periods=mod_clim.sizes["time"], freq="MS", unit="s") + pd.Timedelta(days=15).as_unit("s")
mod_clim = mod_clim.assign_coords(time=time_index)
mod_clim = mod_clim.sel(time=mod_clim.time.isin(obs_clim.time.values))

## CMIP6 1990 - 2100
hist_ds = xr.open_dataset(hist, decode_times=False)[var].chunk({'lat':200, "lon":200})
ssp_ds = xr.open_dataset(ssp, decode_times=False)[var].chunk({'lat':200, "lon":200})
mod_full = xr.concat([hist_ds, ssp_ds], dim="time")
time_index = pd.date_range(start="1850-01-01", periods=mod_full.sizes["time"], freq="MS", unit="s") + pd.Timedelta(days=15).as_unit("s")
mod_full = mod_full.assign_coords(time=time_index)
mod_full = mod_full.sel(time=mod_full.time >= pd.Timestamp("1990-01-01"))

## assign correct CRS to mod_clim and mod_full
mod_clim = mod_clim.rio.write_crs("EPSG:4326") if hasattr(mod_clim, "rio") else mod_clim
mod_full = mod_full.rio.write_crs("EPSG:4326") if hasattr(mod_full, "rio") else mod_full

In [4]:
print(obs_clim)
print(mod_clim)
print(mod_full)

<xarray.DataArray 'tasmax' (time: 600, lat: 1130, lon: 1130)> Size: 3GB
dask.array<getitem, shape=(600, 1130, 1130), dtype=float32, chunksize=(600, 200, 200), chunktype=numpy.ndarray>
Coordinates:
  * lon          (lon) float64 9kB 105.0 105.1 105.1 105.2 ... 161.4 161.4 161.5
  * lat          (lat) float64 9kB -44.97 -44.92 -44.87 ... 11.38 11.43 11.48
    spatial_ref  int64 8B 0
  * time         (time) datetime64[s] 5kB 1940-01-16 1940-02-16 ... 1989-12-16
Attributes:
    standard_name:  air_temperature
    long_name:      Daily Maximum Near-Surface Air Temperatures
    units:          deg_C
<xarray.DataArray 'tasmax' (time: 600, lat: 1130, lon: 1130)> Size: 3GB
dask.array<getitem, shape=(600, 1130, 1130), dtype=float32, chunksize=(600, 200, 200), chunktype=numpy.ndarray>
Coordinates:
  * lon      (lon) float64 9kB 105.0 105.1 105.1 105.2 ... 161.4 161.4 161.5
  * lat      (lat) float64 9kB -44.97 -44.92 -44.87 -44.82 ... 11.38 11.43 11.48
    height   float64 8B ...
  * time     (ti

In [5]:
print(obs_clim.sizes["time"])  # should be 600 (50 years * 12)
print(obs_clim.time.values[0], obs_clim.time.values[-1])

600
1940-01-16T00:00:00 1989-12-16T00:00:00


In [ ]:
# check for zero-variance cells (excluding NaN)
obs_std = obs_clim.std(dim="time").compute()
print("Zero-variance cells (obs):", (obs_std == 0).sum().item())

In [7]:
print(obs_clim.sizes)
print(mod_clim.sizes)
print(mod_full.sizes)
print(mod_full.time.values[:5])
print(mod_full.time.values[-5:])

Frozen({'time': 600, 'lat': 1130, 'lon': 1130})
Frozen({'time': 600, 'lat': 1130, 'lon': 1130})
Frozen({'time': 1332, 'lat': 1130, 'lon': 1130})
['1990-01-16T00:00:00' '1990-02-16T00:00:00' '1990-03-16T00:00:00'
 '1990-04-16T00:00:00' '1990-05-16T00:00:00']
['2100-08-16T00:00:00' '2100-09-16T00:00:00' '2100-10-16T00:00:00'
 '2100-11-16T00:00:00' '2100-12-16T00:00:00']


In [8]:
%%time
# Perform the bias correction
kind = "*" if var == "pr" else "+"
qdm = xsdba.QuantileDeltaMapping.train(
    obs_clim, mod_clim,
    nquantiles=50, group="time.month", kind=kind)
print(qdm)
corrected = qdm.adjust(mod_full, extrapolation="constant", interp="linear")
corrected = corrected.compute() # compute once and hold in RAM

QuantileDeltaMapping(group=Grouper(name='time.month'), kind='+', adapt_freq_thresh=None)
CPU times: user 11h 29min 8s, sys: 1d 2h 36min 19s, total: 1d 14h 5min 28s
Wall time: 2h 41min 31s


/home/dafcluster4/miniconda3/envs/xclim_stable/lib/python3.13/site-packages/numba/np/ufunc/parallel.py:371: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


In [11]:
print(corrected)
# print(corrected.min().values, corrected.max().values, corrected.mean().values)

<xarray.DataArray 'tasmax' (time: 1332, lat: 1130, lon: 1130)> Size: 7GB
array([[[      nan,       nan,       nan, ...,       nan,       nan,
               nan],
        [      nan,       nan,       nan, ...,       nan,       nan,
               nan],
        [      nan,       nan,       nan, ...,       nan,       nan,
               nan],
        ...,
        [32.25347 , 32.23794 , 32.344456, ...,       nan,       nan,
               nan],
        [32.21976 , 32.282333, 32.335903, ...,       nan,       nan,
               nan],
        [32.230606, 32.31302 , 32.314907, ...,       nan,       nan,
               nan]],

       [[      nan,       nan,       nan, ...,       nan,       nan,
               nan],
        [      nan,       nan,       nan, ...,       nan,       nan,
               nan],
        [      nan,       nan,       nan, ...,       nan,       nan,
               nan],
...
        [34.01924 , 33.967796, 33.996346, ...,       nan,       nan,
               nan],
        

In [ ]:
# Averages for plotting
land_mask = obs_clim.isel(time=0).notnull()

mod_clim_mean = mod_clim.where(land_mask).mean(dim=["lat", "lon"]).compute()
mod_full_mean = mod_full.where(land_mask).mean(dim=["lat", "lon"]).compute()
obs_clim_mean = obs_clim.mean(dim=["lat", "lon"]).compute()
corrected_mean = corrected.mean(dim=["lat", "lon"])  # already masked, already in memory

In [ ]:
slices = [
    obs_clim.isel(time=0),
    mod_clim.isel(time=0),
    mod_full.sel(time="1990-01-16"),
    corrected.sel(time="1990-01-16"),
    mod_full.sel(time="2090-01-16"),
    corrected.sel(time="2090-01-16")]
slices = [s.compute() for s in slices]
vmin = min(s.min().item() for s in slices)
vmax = max(s.max().item() for s in slices)

fig, axes = plt.subplots(2, 3, figsize=(16, 10), constrained_layout=True)

p0 = slices[0].plot(ax=axes[0, 0], cmap="RdYlBu_r", vmin=vmin, vmax=vmax, add_colorbar=False, x="lon", y="lat")
axes[0, 0].set_title("obs_clim (Jan 1940)")

p1 = slices[1].plot(ax=axes[0, 1], cmap="RdYlBu_r", vmin=vmin, vmax=vmax, add_colorbar=False, x="lon", y="lat")
axes[0, 1].set_title("mod_clim (Jan 1940)")

axes[0, 2].axis("off")

p2 = slices[2].plot(ax=axes[1, 0], cmap="RdYlBu_r", vmin=vmin, vmax=vmax, add_colorbar=False, x="lon", y="lat")
axes[1, 0].set_title("mod_full (Jan 1990)")

p3 = slices[3].plot(ax=axes[1, 1], cmap="RdYlBu_r", vmin=vmin, vmax=vmax, add_colorbar=False, x="lon", y="lat")
axes[1, 1].set_title("corrected (Jan 1990)")

p4 = slices[4].plot(ax=axes[1, 2], cmap="RdYlBu_r", vmin=vmin, vmax=vmax, add_colorbar=False, x="lon", y="lat")
axes[1, 2].set_title("mod_full (Jan 2090)")

fig.colorbar(p0, ax=axes, orientation="vertical", fraction=0.03, pad=0.04, label=corrected.attrs.get("units", ""))
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

mod_clim_mean.groupby("time.month").mean().plot(
    ax=ax, label="Model - biased (1940-1989, calibration)", color="tab:orange", linestyle="-")

mod_full_mean.sel(time=slice("1990", "2014")).groupby("time.month").mean().plot(
    ax=ax, label="Model - biased (1990-2014)", color="tab:orange", linestyle="--")

corrected_mean.sel(time=slice("1990", "2014")).groupby("time.month").mean().plot(
    ax=ax, label="Model - adjusted (1990-2014)", color="tab:green", linestyle=":")

mod_full_mean.sel(time=slice("2080", "2100")).groupby("time.month").mean().plot(
    ax=ax, label="Model - biased (2080-2100)", color="tab:red", linestyle="--")

corrected_mean.sel(time=slice("2080", "2100")).groupby("time.month").mean().plot(
    ax=ax, label="Model - adjusted (2080-2100)", color="tab:red", linestyle=":")

obs_clim_mean.groupby("time.month").mean().plot(
    ax=ax, label="Reference - obs_clim (1940-1989)", color="tab:blue", linestyle="-")

ax.set_xlabel("Month")
ax.set_ylabel(corrected.attrs.get("units", ""))
ax.set_title("Monthly cycle: model calibration bias vs bias-corrected projection")
ax.legend()
plt.show()

In [9]:
%%time
corrected = corrected.rename(var)
corrected.attrs.update({k: v for k, v in mod_full.attrs.items() if k != "bias_adjustment"})
corrected = corrected.drop_vars("height", errors="ignore")
corrected = corrected.drop_vars("spatial_ref", errors="ignore")
corrected = corrected.transpose("time", "lat", "lon")
corrected.to_netcdf(out_path)

CPU times: user 7.04 s, sys: 16.9 s, total: 24 s
Wall time: 3min 8s
